### Whats next ?

We’ll build a Customer Support Bot and take it to the next level by integrating it with LLMs (Large Language Models) and AI Agents using Python.
Your chatbot will become context-aware, enabling it to understand, respond, and interact more intelligently.

### Problem Statement:
Delivering effective and timely customer support is a challenge for many digital platforms. Users often encounter long wait times, unstructured FAQs, and difficulty navigating help centers. Most support systems fail to understand natural language queries or provide contextual, human-like responses.

To address this, we are building an AI-powered Customer Support Bot that understands user queries, searches relevant knowledge bases and responds conversationally just like a human agent would. Our bot reduces dependency on manual support, improves customer satisfaction and scales efficiently. We’ll bring this to life using LangChain, leveraging Retrieval-Augmented Generation (RAG), tools like web search and vector databases and agents for dynamic reasoning and tool selection.

link: https://colab.research.google.com/drive/1fH1AaTUQ9bgGNa_phpClgpmTZ-YFhbSg#scrollTo=PGZi1bx3JuOP

In [32]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Read API keys
openai_api_key = os.getenv("OPENAI_API_KEY")
open_router_api_key = os.getenv("OPENROUTER_API_KEY")
langsmith_api_key = os.getenv("LANGSMITH_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

# Enable LangChain's tracing to monitor and debug chain executions
os.environ["LANGCHAIN_TRACING_V2"] = "true"

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")

if not open_router_api_key:
    raise ValueError("OPENROUTER_API_KEY not found in .env file")

if not langsmith_api_key:
    raise ValueError("LANGSMITH_API_KEY not found in .env file")

if not tavily_api_key:
    raise ValueError("TAVILY_API_KEY not found in .env file")

print("✓ Environment loaded successfully")
print(f"✓ OpenAI API Key: {openai_api_key[:10]}..." if openai_api_key else "✗ OpenAI API Key not found")
print(f"✓ OpenRouter API Key: {open_router_api_key[:10]}..." if open_router_api_key else "✗ OpenRouter API Key not found")
print(f"✓ LangSmith API Key: {langsmith_api_key[:10]}..." if langsmith_api_key else "✗ LangSmith API Key not found")
print(f"✓ Tavily API Key: {tavily_api_key[:10]}..." if tavily_api_key else "✗ Tavily API Key not found")

✓ Environment loaded successfully
✓ OpenAI API Key: sk-proj-Xo...
✓ OpenRouter API Key: sk-or-v1-f...
✓ LangSmith API Key: lsv2_pt_8d...
✓ Tavily API Key: tvly-rRovS...


In [12]:
from IPython.display import Markdown, display

def print_markdown(text):
    """Display formatted markdown text in Jupyter notebooks"""
    display(Markdown(text))

# Show passed message
print_markdown("""
✅ **All Systems Operational**

- ✓ Environment variables loaded
- ✓ OpenAI API Key configured
- ✓ OpenRouter API Key configured  
- ✓ LangSmith API Key configured

Ready to proceed with LLM operations!
""")


✅ **All Systems Operational**

- ✓ Environment variables loaded
- ✓ OpenAI API Key configured
- ✓ OpenRouter API Key configured  
- ✓ LangSmith API Key configured

Ready to proceed with LLM operations!


In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_retrieval_chain
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [8]:
# 1. Load PDF
loader = PyPDFLoader("Bank_Nifty_Option_Strategies_Booklet.pdf")
documents = loader.load()

# 2. Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(documents)

# 3. Create embeddings and vector store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents, embeddings)

In [10]:
documents[:2]  # Display first two documents

[Document(metadata={'producer': 'Corel PDF Engine Version 15.1.0.588', 'creator': 'CorelDRAW X5', 'creationdate': '2012-08-06T15:48:31+05:30', 'author': 'Marketing Support', 'moddate': '2012-08-06T15:54:49+05:30', 'title': 'Bank Nifty Option Strategies Booklet', 'source': 'Bank_Nifty_Option_Strategies_Booklet.pdf', 'total_pages': 28, 'page': 0, 'page_label': '1'}, page_content='Bank Nifty\nOptions\nStrategies'),
 Document(metadata={'producer': 'Corel PDF Engine Version 15.1.0.588', 'creator': 'CorelDRAW X5', 'creationdate': '2012-08-06T15:48:31+05:30', 'author': 'Marketing Support', 'moddate': '2012-08-06T15:54:49+05:30', 'title': 'Bank Nifty Option Strategies Booklet', 'source': 'Bank_Nifty_Option_Strategies_Booklet.pdf', 'total_pages': 28, 'page': 1, 'page_label': '2'}, page_content='')]

In [11]:
llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    temperature=0.7, 
    openai_api_key=open_router_api_key, 
    model="google/gemma-3-27b-it:free")

output_parser = StrOutputParser()
prompt = ChatPromptTemplate.from_template(
    "You are a option trader expert. Based on the following document excerpts, answer the question and do not invent information:\n\n"
    "{context}\n\n"
    "Question: {input}\n"
    "Answer in detail:", output_parser=output_parser
)

document_chain = create_stuff_documents_chain(llm, prompt)
retriever = vectorstore.as_retriever()
retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [17]:
response = retrieval_chain.invoke({"input": "what is best strategy for bank nifty options trading?"})
print_markdown(response['answer'])

Based on the provided document excerpts, there isn't a single "best" strategy explicitly stated. Instead, the document outlines several strategies with different risk/reward profiles. Here's a breakdown of what's presented:

1. **Selling Put Options (Bearish Strategy):**
   - **Profit:** When Bank Nifty *doesn't* go down (and the option expires unexercised).
   - **Loss:** When Bank Nifty *does* go down (and the option is exercised).
   - The payoff graph shows potential for profit if Bank Nifty stays above a certain level (around 8200-8400 in the example).  However, losses can be substantial if Bank Nifty falls significantly.

2. **Buying Call Options (Bullish Strategy):**
   - **Profit:** When Bank Nifty closes *above* the strike price on expiry.
   - **Loss:** When Bank Nifty closes *below* the strike price on expiry.
   - The document provides an example of buying a call option with a strike price of 8900 and a premium of 200, with a break-even point at 8700. The payoff graph shows potential for profit as Bank Nifty rises above 8900.

3. **Combined Strategy (Buy ITM Put & Sell OTM Put):**
   - The document mentions an example of buying one In-The-Money (ITM) Put option *and* selling one Out-of-The-Money (OTM) Put option. The details of the payoff are not fully explained, but it suggests a more complex strategy potentially aiming to limit risk or reduce the cost of the put option.

4. **Selling Call Options:**
   - The document shows a payoff graph for selling call options. Profit is made when Bank Nifty stays below the strike price, and loss is incurred when it rises above.

**In conclusion:**

The "best" strategy depends entirely on your outlook for Bank Nifty. 

*   If you are **bullish**, buying call options is presented as a potential strategy.
*   If you are **bearish**, selling put options is presented as a potential strategy.
*   The combined put strategy is a more nuanced approach, but its specifics aren't fully detailed in the excerpts.



It's important to note that options trading involves risk, and the document highlights the potential for both profit and loss with each strategy. The provided information is limited, and a thorough understanding of options trading principles is crucial before implementing any strategy.

In [34]:
#from langchain.tools import create_retriever_tool
#from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_core.tools import Tool, tool
from langchain_openai import ChatOpenAI
from langchain.agents import AgentState, create_agent
from langchain.messages import MessageLikeRepresentation
from langchain.chat_models import init_chat_model
from langchain_community.tools.tavily_search import TavilySearchResults

search = TavilySearchResults()

@tool
def safe_tavily_search(query: str = "options trading strategies"):
    """Safe tavily search. Use for any web search."""
    if not query or query.strip() == "":
        return "No query provided to search tool."
    return search.invoke({"query": query})

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = retriever.vectorstore.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

retriever_tool = Tool(
    name="bank_nifty_search",
    func=retriever.invoke,
    description="Search for information about Bank Nifty option strategies from the PDF"
)

tools = [retrieve_context, safe_tavily_search]


# If desired, specify custom instructions
prompt = (
    "Your an option trader expert and You have access to a tool that retrieves context from option startergy document. "
    "Use the tool to help answer user queries."
)
#openai:gpt-5-mini
agent = create_agent(model="openai:gpt-4.1-mini", tools=tools, system_prompt=prompt)

In [36]:
query = "What current nifty trading level and how to use it for options trading?"
for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What current nifty trading level and how to use it for options trading?
================================== Ai Message ==================================
Tool Calls:
  safe_tavily_search (call_4m5KUYyVrRw7FLfZyzrb0cXD)
 Call ID: call_4m5KUYyVrRw7FLfZyzrb0cXD
  Args:
    query: current nifty trading level
  retrieve_context (call_navu0W2AG1e45BGeRk9iHnnd)
 Call ID: call_navu0W2AG1e45BGeRk9iHnnd
  Args:
    query: how to use nifty trading level for options trading
================================= Tool Message =================================
Name: retrieve_context

Source: {'page_label': '1', 'title': 'Bank Nifty Option Strategies Booklet', 'creator': 'CorelDRAW X5', 'creationdate': '2012-08-06T15:48:31+05:30', 'moddate': '2012-08-06T15:54:49+05:30', 'source': 'Bank_Nifty_Option_Strategies_Booklet.pdf', 'total_pages': 28, 'producer': 'Corel PDF Engine Version 15.1.0.588', 'author': 'Marketing Support', 'pa